### 1. Initial Setup: Clean environment

In [59]:
import os
import shutil

# Ensure the Python interpreter is in /content
os.chdir('/content')
print(f"Current working directory set to: {os.getcwd()}")

# Remove Colab's default sample_data directory
if os.path.exists('/content/sample_data'):
    shutil.rmtree('/content/sample_data')
    print("Removed /content/sample_data")

# Remove previously cloned Semantic-Correspondence repository if it exists
if os.path.exists('/content/Semantic-Correspondence'):
    shutil.rmtree('/content/Semantic-Correspondence')
    print("Removed /content/Semantic-Correspondence")

# Remove previously cloned data if it exists outside
if os.path.exists('/content/data'):
    shutil.rmtree('/content/data')
    print("Removed /content/data")

print("Environment cleaned.")

Current working directory set to: /content
Removed /content/Semantic-Correspondence
Environment cleaned.


### 2. Clone the Semantic-Correspondence Repository

In [60]:
import shutil
import os

%cd /content

# Remove previously cloned Semantic-Correspondence repository if it exists
if os.path.exists('/content/Semantic-Correspondence'):
    shutil.rmtree('/content/Semantic-Correspondence')
    print("Removed existing /content/Semantic-Correspondence directory.")

!git clone https://github.com/lucaosti/Semantic-Correspondence && echo "Repository cloned successfully." || echo "Error: Could not clone repository."

/content/Semantic-Correspondence
Cloning into 'Semantic-Correspondence'...
remote: Enumerating objects: 140, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (108/108), done.
Receiving objects: 100% (140/140), 144.33 KiB | 5.15 MiB/s, done.
remote: Total 140 (delta 32), reused 134 (delta 32), pack-reused 0 (from 0)
Resolving deltas: 100% (32/32), done.
Repository cloned successfully.


### 3. Change directory and Download Dataset

In [61]:
# Change current directory to Semantic-Correspondence
%cd /content/Semantic-Correspondence

# Create the data directory if it doesn't exist
if not os.path.exists('data'):
    os.makedirs('data')

# Download the dataset into the 'data/' directory
dataset_url = "https://cvlab.postech.ac.kr/research/SPair-71k/data/SPair-71k.tar.gz"
dataset_path_in_repo = os.path.join('data', 'SPair-71k.tar.gz')

if not os.path.exists(dataset_path_in_repo):
    print(f"Downloading {dataset_url} to {dataset_path_in_repo}...")
    !wget -P data/ {dataset_url}
    print("Dataset downloaded.")
else:
    print("Dataset already exists in data/ directory.")

/content/Semantic-Correspondence
--2026-03-22 09:05:47--  https://cvlab.postech.ac.kr/research/SPair-71k/data/SPair-71k.tar.gz
Resolving cvlab.postech.ac.kr (cvlab.postech.ac.kr)... 141.223.85.126
Connecting to cvlab.postech.ac.kr (cvlab.postech.ac.kr)|141.223.85.126|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 226961117 (216M) [application/x-gzip]
Saving to: ‘data/SPair-71k.tar.gz’

SPair-71k.tar.gz    100%[===================>] 216.45M  11.2MB/s    in 21s     

2026-03-22 09:06:08 (10.5 MB/s) - ‘data/SPair-71k.tar.gz’ saved [226961117/226961117]

Dataset downloaded.


### 5. Configurazione da YAML e Import per il Workflow Programmatico

In [62]:
import os
import yaml
import torch # Import torch to use torch.cuda.is_available()

# Carica il file di configurazione YAML
config_path = os.path.join('/content/Semantic-Correspondence', 'config.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# --- Mappa le configurazioni YAML alle variabili Python --- (o usale direttamente da 'config')
REPO_ROOT = config['paths']['repo_root']
SPAIR_ROOT = config['paths']['spair_root']
CHECKPOINT_DIR = config['paths']['checkpoint_dir']

# Crea la directory dei checkpoint se non esiste
if not os.path.exists(CHECKPOINT_DIR):
    os.makedirs(CHECKPOINT_DIR)

SAM_CHECKPOINT = config['finetune']['sam_checkpoint']
DINOV2_WEIGHTS = config['finetune']['dinov2_weights']
DINOV3_WEIGHTS = config['finetune']['dinov3_weights']

LAST_BLOCKS = config['finetune']['last_blocks']
FT_EPOCHS = config['finetune']['epochs']
FT_PATIENCE = config['finetune']['patience']

LORA_RANK = config['lora']['rank']
LORA_EPOCHS = config['lora']['epochs']
LORA_PATIENCE = config['lora']['patience']

PREPROCESS = config['runtime']['preprocess']
IMAGE_HEIGHT = config['runtime']['image_height']
IMAGE_WIDTH = config['runtime']['image_width']

# Auto-detect device if 'null'
if config['runtime']['device'] == None:
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
else:
    DEVICE = config['runtime']['device']

# Auto-detect num_workers if '-1'
if config['runtime']['num_workers'] == -1:
    NUM_WORKERS = os.cpu_count() // 2
else:
    NUM_WORKERS = config['runtime']['num_workers']

EVAL_SPLIT = 'test' # Default for final eval, can be changed via experiments
EVAL_LIMIT = config['runtime']['limit_pairs']
EVAL_ALPHAS = config['runtime']['alphas']
WSA_WINDOW = config['runtime']['wsa_window']
WSA_TEMPERATURE = config['runtime']['wsa_temperature']
LOG_BATCH_INTERVAL = 2500 # Not in YAML, keeping a default

# Workflow Toggles from YAML
RUN_VERIFY_DATASET = config['workflow_toggles']['run_verify_dataset']
TRAIN_FINETUNE = config['workflow_toggles']['train_finetune']
TRAIN_LORA = config['workflow_toggles']['train_lora']
RUN_EVAL_BASELINE = config['workflow_toggles']['run_eval_baseline']
RUN_EVAL_BASELINE_WSA = config['workflow_toggles']['run_eval_baseline_wsa']
RUN_EVAL_FINETUNED_CHECKPOINT = config['workflow_toggles']['run_eval_finetuned_checkpoint']
RUN_EVAL_LORA_CHECKPOINT = config['workflow_toggles']['run_eval_lora_checkpoint']
RUN_EXPORT_METRICS_TABLES = config['workflow_toggles']['run_export_metrics_tables']
PIPELINE_RESUME = config['workflow_toggles']['pipeline_resume']
RUN_PYTEST = config['workflow_toggles']['run_pytest']
PRINT_JUPYTER_NOTEBOOK_HINT = config['workflow_toggles']['print_jupyter_notebook_hint']

print("Configurazione caricata da config.yaml.")
print(f"Device selezionato: {DEVICE}")
print(f"Numero di workers: {NUM_WORKERS}")

FileNotFoundError: [Errno 2] No such file or directory: '/content/Semantic-Correspondence/config.yaml'

In [63]:
import os

config_content = """
paths:
  repo_root: /content/Semantic-Correspondence
  spair_root: /content/Semantic-Correspondence/data/SPair-71k
  checkpoint_dir: /content/Semantic-Correspondence/checkpoints
  output_dir: /content/Semantic-Correspondence/runs/notebook_exports

runtime:
  device: null
  num_workers: -1
  preprocess: FIXED_RESIZE
  image_height: 784
  image_width: 784
  limit_pairs: 10
  alphas: [0.05, 0.1, 0.15]
  wsa_window: 5
  wsa_temperature: 0.05

finetune:
  backbone: dinov2_vitb14
  last_blocks: 2
  epochs: 5
  patience: 3
  learning_rate: 0.0001
  weight_decay: 0.00001
  resume: false
  dinov2_weights: null
  dinov3_weights: null
  sam_checkpoint: /content/Semantic-Correspondence/sam_vit_b.pth

lora:
  backbone: dinov2_vitb14
  last_blocks: 2
  epochs: 5
  patience: 3
  learning_rate: 0.0001
  weight_decay: 0.00001
  resume: false
  rank: 16
  alpha: 16

workflow_toggles:
  run_verify_dataset: true
  train_finetune: [true, false, false]
  train_lora: [false, false, false]
  run_eval_baseline: [true, false, false]
  run_eval_baseline_wsa: [false, false, false]
  run_eval_finetuned_checkpoint: [false, false, false]
  run_eval_lora_checkpoint: [false, false, false]
  run_export_metrics_tables: false
  pipeline_resume: false
  run_pytest: false
  print_jupyter_notebook_hint: false
"""

config_path = '/content/Semantic-Correspondence/config.yaml'
os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, 'w') as f:
    f.write(config_content)

print(f"Config file created at {config_path}")

Config file created at /content/Semantic-Correspondence/config.yaml


In [64]:
import os
!cd /content/Semantic-Correspondence && bash scripts/download_sam_vit_b.sh
os.environ['SAM_CHECKPOINT'] = '/content/Semantic-Correspondence/sam_vit_b.pth'
print("SAM Checkpoint downloaded and environment variable set.")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  357M  100  357M    0     0  64.5M      0  0:00:05  0:00:05 --:--:-- 70.5M
Done.
SAM Checkpoint downloaded and environment variable set.


In [65]:
!python /content/Semantic-Correspondence/scripts/run_pipeline.py --config /content/Semantic-Correspondence/config.yaml

[2026-03-22 09:07:11 UTC] Adaptive hardware: device=cpu num_workers=2 cpu_count=2 platform=linux
[2026-03-22 09:07:11 UTC] pipeline_state: no prior file (will write /content/Semantic-Correspondence/runs/pipeline_state.json).
[2026-03-22 09:07:11 UTC] Tracing: structured log /content/Semantic-Correspondence/runs/logs/stage_events.jsonl | state file /content/Semantic-Correspondence/runs/pipeline_state.json
[2026-03-22 09:07:11 UTC] [STAGE] action=start stage_id=verify_dataset
[2026-03-22 09:07:11 UTC] --- Running: /usr/bin/python3 /content/Semantic-Correspondence/scripts/verify_dataset.py
[2026-03-22 09:07:16 UTC] ERROR: Missing SPair root: /content/Semantic-Correspondence/data/SPair-71k
[2026-03-22 09:07:16 UTC] Unpack SPair-71k under data/SPair-71k/ or set SPAIR_ROOT.
[2026-03-22 09:07:17 UTC] [STAGE] action=fail stage_id=verify_dataset exit_code=2


In [ ]:
# Cell 3: Import Principali

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from data.dataset import PreprocessMode, SPair71kPairDataset, spair_collate_fn
from evaluation import EvalRunSpec, metrics_rows_for_table, run_comparison_batch, run_spair_pck_eval
from models.dinov2.backbone import build_dinov2_vit_b14
from models.dinov3.backbone import build_dinov3_vit_b16
from models.sam.backbone import build_sam_vit_b_image_encoder
from training import (
    EarlyStopping,
    collect_trainable_parameter_groups,
    correspondence_gaussian_loss_dino_vit,
    correspondence_gaussian_loss_sam,
    freeze_all,
    unfreeze_last_transformer_blocks,
)
from training.engine import train_model_finetune # Importa la funzione di training
from models.common.lora import apply_lora_to_last_blocks_mlp, apply_lora_to_last_blocks_mlp_sam, lora_trainable_parameters
from utils.hardware import (
    apply_accelerator_throughput_tweaks,
    dataloader_extra_kwargs,
    loader_worker_init_for_device,
    maybe_tune_threads_for_cpu_device,
    pin_memory_for,
    resolve_device_str,
    resolve_num_workers,
)

print("Moduli principali importati.")

In [ ]:
print("Starting training (programmatic section - this part will be handled by external scripts).")

In [ ]:
# This cell is superseded by 'start_dashboard.sh' as per operational instructions.
# print("Attempting to launch the dashboard. Look for a public URL in the output.")
# !python scripts/pipeline_dashboard.py
# print("Dashboard script executed. Check output for URL.")

In [ ]:
print("Installing PyYAML...")
!pip install pyyaml
print("PyYAML installed.")

In [ ]:
import os

config_content = """
# config.yaml
paths:
  repo_root: /content/Semantic-Correspondence
  spair_root: /content/Semantic-Correspondence/data/SPair-71k
  checkpoint_dir: /content/Semantic-Correspondence/checkpoints
  output_dir: /content/Semantic-Correspondence/runs/notebook_exports

runtime:
  device: null # auto-detect, can be 'cuda', 'mps', 'cpu'
  num_workers: -1 # auto-detect, or set a fixed integer
  preprocess: FIXED_RESIZE
  image_height: 784
  image_width: 784
  limit_pairs: 10 # 0 for full dataset, small number for quick tests
  alphas: [0.05, 0.1, 0.15]
  wsa_window: 5
  wsa_temperature: 0.05

finetune:
  backbone: dinov2_vitb14 # one of dinov2_vitb14, dinov3_vitb16, sam_vit_b
  last_blocks: 2
  epochs: 200
  patience: 10
  learning_rate: 0.0001
  weight_decay: 0.00001
  resume: false
  # Paths to official weights, if needed. DINOv2 often loads automatically.
  dinov2_weights: null
  dinov3_weights: null
  sam_checkpoint: /content/Semantic-Correspondence/sam_vit_b.pth # Path after downloading

lora:
  backbone: dinov2_vitb14
  last_blocks: 2
  epochs: 100
  patience: 10
  learning_rate: 0.0001
  weight_decay: 0.00001
  resume: false
  rank: 16
  alpha: 16

experiments: # Example structure, will be used by pipeline scripts
  - name: baseline_dinov2
    backbone: dinov2_vitb14
    split: test
    # checkpoint: path/to/checkpoint.pth # Uncomment for specific checkpoints

workflow_toggles:
  run_verify_dataset: true
  train_finetune: [true, false, false] # Abilita fine-tuning per DINOv2, DINOv3, SAM
  train_lora: [false, false, false]
  run_eval_baseline: [true, false, false] # Abilita eval baseline per DINOv2
  run_eval_baseline_wsa: [false, false, false]
  run_eval_finetuned_checkpoint: [false, false, false]
  run_eval_lora_checkpoint: [false, false, false]
  run_export_metrics_tables: false
  pipeline_resume: false
  run_pytest: false
  print_jupyter_notebook_hint: false
"""

config_path = os.path.join('/content/Semantic-Correspondence', 'config.yaml')
with open(config_path, 'w') as f:
    f.write(config_content)

print(f"config.yaml created at {config_path}")

In [ ]:
print("Downloading SAM ViT-B checkpoint...")
%cd /content/Semantic-Correspondence
!bash scripts/download_sam_vit_b.sh
print("SAM ViT-B checkpoint downloaded.")

import tarfile
import os
import shutil

data_dir = '/content/Semantic-Correspondence/data/'
dataset_archive_path = os.path.join(data_dir, 'SPair-71k.tar.gz')
target_extracted_dir = os.path.join(data_dir, 'SPair-71k')

if os.path.exists(dataset_archive_path):
    print(f'Extracting {dataset_archive_path} to {data_dir}...')
    with tarfile.open(dataset_archive_path, 'r:gz') as tar:
        tar.extractall(path=data_dir)
    print('Extraction complete.')
    if os.path.exists(target_extracted_dir):
        os.remove(dataset_archive_path)
        print(f'Archive removed. Dataset ready in {target_extracted_dir}')

if os.path.exists(target_extracted_dir):
    print('Starting Pipeline...')
    !python /content/Semantic-Correspondence/scripts/run_pipeline.py --config /content/Semantic-Correspondence/config.yaml
else:
    print(f'Error: {target_extracted_dir} not found. Please check download step.')

In [ ]:
import tarfile

data_dir = '/content/Semantic-Correspondence/data/'
dataset_archive_path = os.path.join(data_dir, 'SPair-71k.tar.gz')
target_extracted_dir = os.path.join(data_dir, 'SPair-71k')

if os.path.exists(target_extracted_dir):
    print(f"Removing existing extracted dataset directory: {target_extracted_dir}")
    shutil.rmtree(target_extracted_dir)

if os.path.exists(dataset_archive_path):
    print(f"Unzipping {dataset_archive_path} to {data_dir}...")
    try:
        with tarfile.open(dataset_archive_path, "r:gz") as tar:
            tar.extractall(path=data_dir)
        print("Dataset unzipped successfully.")

        # Remove the tar.gz file after extraction
        os.remove(dataset_archive_path)
        print(f"Removed {dataset_archive_path}.")
    except Exception as e:
        print(f"Error unzipping dataset: {e}")
else:
    print(f"Dataset archive not found at {dataset_archive_path}. Skipping extraction.")

# Verify content after extraction
if os.path.exists(target_extracted_dir) and len(os.listdir(target_extracted_dir)) > 0:
    print(f"Content found in {target_extracted_dir}.")
else:
    print(f"Warning: {target_extracted_dir} appears to be empty or missing content after extraction.")

### 5. Install Dependencies

In [ ]:
# Change to the repository directory before installing dependencies
%cd /content/Semantic-Correspondence

# Add the current working directory (project root) to the Python path
# This ensures that modules like 'data' can be imported correctly from 'scripts/'
import os
os.environ['PYTHONPATH'] = os.getcwd() + os.pathsep + os.environ.get('PYTHONPATH', '')

# Install the package in editable mode as per documentation
!pip install -e ".[notebook]"
print("Dependencies installed in editable mode.")
print(f"PYTHONPATH set to: {os.environ['PYTHONPATH']}")

### 8. Run Pipeline (Train and Evaluate)

In [ ]:
print("Starting pipeline...")
# Esegui lo script del pipeline utilizzando il file di configurazione YAML
# Questo script leggerà la configurazione e i toggles per decidere cosa eseguire.
!python scripts/run_pipeline.py --config /content/Semantic-Correspondence/config.yaml
print("Pipeline finished.")

### 9. Avvio Dashboard

In [ ]:
print("Attempting to launch the dashboard. Look for a public URL in the output.")
# Execute the dashboard start script. This may provide a public URL.
!bash scripts/start_dashboard.sh
print("Dashboard script executed. Check output for URL.")